In [1]:
# ================================================
# ✅ 项目总结：Statham 名言机器人
# ================================================
# 📌 项目目标：
# 开发一个基于 Telegram 平台的聊天机器人，结合随机名言与贴纸互动功能，
# 提升用户体验，锻炼使用 python-telegram-bot v20+ 库进行异步编程的能力。

# 🎯 实现功能：
# 1. /start 命令：显示欢迎语与交互按钮；
# 2. /quote 命令：返回一条随机名言和一张贴纸；
# 3. Inline 按钮：用户可点击“新名言”再次获取内容；
# 4. 每次回复均包含一句名言 + 一张贴纸 + Inline 按钮；
# 5. 数据部分使用 QUOTES 和 STICKERS 列表进行本地管理，无需联网调用API。

# 🧠 技术实现：
# - 使用 python-telegram-bot v20.3 库；
# - 使用异步函数 async/await 编写响应逻辑；
# - 利用 InlineKeyboardMarkup 实现交互按钮；
# - 数据随机选取使用 random 模块；
# - 环境部署建议使用 Anaconda + Jupyter Notebook。

# 💡 创新与拓展建议：
# - 可接入 OpenAI/Gemini 接口生成名言；
# - 可接入数据库实现收藏夹与评分功能；
# - 可添加每日定时推送任务。

# 🚀 运行方式：
# 1. 使用 pip 安装依赖；
# 2. 替换 TOKEN 为你在 BotFather 创建的机器人密钥；
# 3. 运行最后一单元启动机器人，手机端体验功能。

# 作者：NeonChen
# 时间：2025年6月21日
# ================================================

In [2]:
# 安装 python-telegram-bot v20.3（仅需运行一次）
!pip install -q python-telegram-bot==20.3

'pip' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [3]:
# import logging
# logging.basicConfig(level=logging.INFO)

In [4]:
import os
import random
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import Application, CommandHandler, CallbackQueryHandler, ContextTypes

In [5]:
QUOTES = [
    "Sometimes you have to take a step back to move forward.",
    "I don't believe in luck. I believe in hard work.",
    "You never know how strong you are until being strong is the only choice you have.",
    "Success is the sum of small efforts repeated day in and day out.",
    "The only way to do great work is to love what you do.",
    "Your time is limited, don't waste it living someone else's life.",
    "If you want something you've never had, you must be willing to do something you've never done.",
    "The harder the battle, the sweeter the victory.",
    "Dream big and dare to fail.",
    "What defines us is how well we rise after falling.",
    "The best revenge is massive success.",
    "Don't count the days, make the days count.",
    "It always seems impossible until it's done.",
    "The only limit to our realization of tomorrow is our doubts of today.",
    "Life is 10% what happens to us and 90% how we react to it.",
    "Believe you can and you're halfway there.",
    "Strength does not come from winning. Your struggles develop your strengths.",
    "The only place where success comes before work is in the dictionary.",
    "If you're going through hell, keep going.",
    "The difference between ordinary and extraordinary is that little extra."
]

STICKERS = [
"CAACAgIAAxkBAAMUaFbRgljTQ9Aw6cxuZbP_jNPmhtkAAhsAA8A2TxN9vYqDy0RNajYE"
]

In [6]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    keyboard = [[InlineKeyboardButton("随机名言", callback_data='random_quote')]]
    reply_markup = InlineKeyboardMarkup(keyboard)
    await update.message.reply_text(
        "🔥 欢迎使用 Stetham 名言机器人！\n"
        "输入 /quote 获取随机名言+贴纸\n"
        "或点击下方按钮：",
        reply_markup=reply_markup
    )

async def quote(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await send_random_quote(update, context)

async def button_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    await send_random_quote(update, context, query.message)

In [7]:
async def send_random_quote(update, context, message=None):
    quote_text = random.choice(QUOTES)
    sticker_id = random.choice(STICKERS)

    keyboard = [[
        InlineKeyboardButton("新名言", callback_data='random_quote'),
        InlineKeyboardButton("源码", url='https://github.com')
    ]]
    reply_markup = InlineKeyboardMarkup(keyboard)

    if message:
        await message.reply_sticker(sticker_id)
        await message.reply_text(f"“{quote_text}”\n\n- Jason Statham", reply_markup=reply_markup)
    else:
        await update.message.reply_sticker(sticker_id)
        await update.message.reply_text(f"“{quote_text}”\n\n- Jason Statham", reply_markup=reply_markup)

In [8]:
TOKEN = "7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc" 

application = Application.builder().token(TOKEN).build()


application.add_handler(CommandHandler("start", start))
application.add_handler(CommandHandler("quote", quote))
application.add_handler(CallbackQueryHandler(button_handler))

print("机器人已启动...")
# application.run_polling()

机器人已启动...


In [9]:
from telegram.ext import MessageHandler, filters

async def show_file_id(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if update.message.sticker:
        await update.message.reply_text(f"✅ 收到贴纸 file_id：\n{update.message.sticker.file_id}")

application.add_handler(MessageHandler(filters.Sticker.ALL, show_file_id))

In [10]:
import asyncio

async def main():
    await application.initialize()
    await application.start()
    await application.updater.start_polling(drop_pending_updates=True)
    print("🤖 机器人已启动（Notebook兼容模式）")
    # await application.updater.start_polling()
    # 注意：不使用 `run_polling()`，而是分步控制
    # 保持运行直到手动停止
    await asyncio.Event().wait()

await main()

INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/getMe "HTTP/1.1 200 OK"
INFO:telegram.ext.Application:Application started
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/deleteWebhook "HTTP/1.1 200 OK"


🤖 机器人已启动（Notebook兼容模式）


INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/sendMessage "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/getUpdates "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/answerCallbackQuery "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.org/bot7759014904:AAFbtXGHtDJjVwQFVLf7F_XdyHZw9ufB0Tc/sendSticker "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.telegram.

CancelledError: 